# Estimacion de bandwidths

Este notebook calcula bandwidths globales y tambien estima el mejor bandwidth por kernel mediante validacion cruzada.

In [ ]:
from pathlib import Path
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import gaussian_kde
from sklearn.model_selection import KFold
from sklearn.neighbors import KernelDensity

cwd = Path.cwd().resolve()
notebook_dir = cwd if cwd.name == "Kernel_Tests" else cwd / "Kernel_Tests"


def resolve_data_path(relative_path):
    data_path = Path(relative_path)
    if data_path.is_absolute():
        return data_path
    candidate = notebook_dir / data_path
    if candidate.exists():
        return candidate.resolve()
    return (notebook_dir.parent / data_path).resolve()


In [ ]:
def load_otu_positives(data_path):
    file_path = resolve_data_path(data_path)
    if not file_path.exists():
        raise FileNotFoundError("No se encontro el archivo de datos configurado.")

    df = pd.read_csv(file_path)
    values = df.select_dtypes(include=[np.number]).to_numpy(dtype=float).ravel()
    values = values[np.isfinite(values)]
    positives = values[values > 0]
    return positives, df.shape


def suggest_grid_size(n_values):
    if n_values <= 250_000:
        return 1000
    if n_values <= 1_000_000:
        return 1500
    return 2000


def make_log_grid(values, grid_size, bandwidth):
    positives = np.asarray(values, dtype=float)
    positives = positives[np.isfinite(positives)]
    positives = positives[positives > 0]
    if positives.size == 0:
        raise ValueError("Se requiere al menos un valor positivo.")
    if bandwidth <= 0:
        raise ValueError("El bandwidth debe ser positivo.")

    lower = max(float(np.min(positives)) * 1e-3, 1e-12)
    upper = float(np.max(positives) + 8.0 * bandwidth)
    return np.logspace(np.log10(lower), np.log10(upper), int(grid_size))


In [ ]:
def prepare_values(values):
    arr = np.asarray(values, dtype=float).ravel()
    arr = arr[np.isfinite(arr)]
    if arr.size < 2:
        raise ValueError("Se requieren al menos 2 valores finitos.")
    if np.nanstd(arr, ddof=1) <= 0:
        raise ValueError("No se puede estimar bandwidth con varianza cero.")
    return arr


def scott_h(values):
    values = prepare_values(values)
    std = float(np.std(values, ddof=1))
    return float(gaussian_kde(values, bw_method="scott").factor * std)


def silverman_h(values):
    values = prepare_values(values)
    std = float(np.std(values, ddof=1))
    return float(gaussian_kde(values, bw_method="silverman").factor * std)


def cv_loglik(
    values,
    kernel="gaussian",
    cv_folds=5,
    n_subsample=5000,
    n_bw=25,
    bw_lo_factor=0.02,
    bw_hi_factor=3.0,
    seed=42,
):
    values = prepare_values(values)
    rng = np.random.default_rng(seed)
    k = min(int(n_subsample), len(values))
    folds = min(int(cv_folds), k)
    if folds < 2:
        raise ValueError("Se requieren al menos 2 observaciones para validacion cruzada.")

    sample = values[rng.choice(len(values), size=k, replace=False)].reshape(-1, 1)
    bw_scott_abs = scott_h(values)
    bw_min = max(0.5, bw_scott_abs * bw_lo_factor)
    bw_max = bw_scott_abs * bw_hi_factor
    bw_grid = np.geomspace(bw_min, bw_max, int(n_bw))

    kf = KFold(n_splits=folds, shuffle=True, random_state=seed)
    scores = np.zeros(len(bw_grid))
    for i, bw in enumerate(bw_grid):
        fold_scores = []
        for train_idx, val_idx in kf.split(sample):
            kde = KernelDensity(kernel=kernel, bandwidth=bw, breadth_first=False)
            kde.fit(sample[train_idx])
            fold_scores.append(kde.score(sample[val_idx]))
        scores[i] = float(np.mean(fold_scores))

    best_idx = int(np.argmax(scores))
    return float(bw_grid[best_idx]), bw_grid, scores


def estimate_cv_by_kernel(values, kernels, cv_folds, n_subsample, n_bw):
    rows = []
    score_curves = {}
    for kernel in kernels:
        best_bw, bw_grid, scores = cv_loglik(
            values,
            kernel=kernel,
            cv_folds=cv_folds,
            n_subsample=n_subsample,
            n_bw=n_bw,
        )
        best_idx = int(np.argmax(scores))
        rows.append({
            "kernel": kernel,
            "bandwidth_cv": best_bw,
            "score_cv": float(scores[best_idx]),
        })
        score_curves[kernel] = (bw_grid, scores)
    return pd.DataFrame(rows), score_curves


In [ ]:
KERNELS = ("gaussian", "epanechnikov", "tophat", "exponential", "linear", "cosine")
COLOR_MAP = {
    "gaussian": "#4c78a8",
    "epanechnikov": "#e45756",
    "tophat": "#f58518",
    "exponential": "#54a24b",
    "linear": "#b279a2",
    "cosine": "#9d755d",
}
FINITE_FAST_KERNELS = {"epanechnikov", "tophat", "linear", "cosine"}


def positive_mass(pdf, x_grid):
    return float(np.trapezoid(pdf, x_grid))


def normalize_conditional(pdf, x_grid):
    mass = positive_mass(pdf, x_grid)
    if not np.isfinite(mass) or mass <= 0:
        raise ValueError("La densidad KDE tiene masa no positiva o no finita.")
    return pdf / mass, mass


def mode_kde(pdf, x_grid):
    return float(x_grid[int(np.argmax(pdf))])


def evaluate_kde_reference(data, x_grid, bandwidth, kernel):
    kde = KernelDensity(kernel=kernel, bandwidth=bandwidth)
    kde.fit(np.asarray(data, dtype=float).reshape(-1, 1))
    return np.exp(kde.score_samples(np.asarray(x_grid, dtype=float).reshape(-1, 1)))


def evaluate_kde_gaussian_fast(data, x_grid, bandwidth):
    kde = KernelDensity(kernel="gaussian", bandwidth=bandwidth, breadth_first=False)
    kde.fit(np.asarray(data, dtype=float).reshape(-1, 1))
    return np.exp(kde.score_samples(np.asarray(x_grid, dtype=float).reshape(-1, 1)))


def sorted_prefix(data):
    sorted_data = np.sort(np.asarray(data, dtype=float))
    prefix_1 = np.concatenate(([0.0], np.cumsum(sorted_data)))
    prefix_2 = np.concatenate(([0.0], np.cumsum(sorted_data * sorted_data)))
    return sorted_data, prefix_1, prefix_2


def evaluate_finite_support_fast(data, x_grid, bandwidth, kernel):
    sorted_data, prefix_1, prefix_2 = sorted_prefix(data)
    n = float(sorted_data.size)
    x = np.asarray(x_grid, dtype=float)
    left = np.searchsorted(sorted_data, x - bandwidth, side="left")
    right = np.searchsorted(sorted_data, x + bandwidth, side="right")
    count = (right - left).astype(float)

    if kernel == "tophat":
        return 0.5 * count / (n * bandwidth)

    if kernel == "epanechnikov":
        sum_1 = prefix_1[right] - prefix_1[left]
        sum_2 = prefix_2[right] - prefix_2[left]
        sq_dist = count * x * x - 2.0 * x * sum_1 + sum_2
        return 0.75 * (count - sq_dist / (bandwidth * bandwidth)) / (n * bandwidth)

    if kernel == "linear":
        mid = np.searchsorted(sorted_data, x, side="right")
        mid = np.minimum(np.maximum(mid, left), right)
        left_count = (mid - left).astype(float)
        right_count = (right - mid).astype(float)
        left_sum = prefix_1[mid] - prefix_1[left]
        right_sum = prefix_1[right] - prefix_1[mid]
        abs_dist = x * left_count - left_sum + right_sum - x * right_count
        return (count - abs_dist / bandwidth) / (n * bandwidth)

    if kernel == "cosine":
        c = np.pi / (2.0 * bandwidth)
        prefix_cos = np.concatenate(([0.0], np.cumsum(np.cos(c * sorted_data))))
        prefix_sin = np.concatenate(([0.0], np.cumsum(np.sin(c * sorted_data))))
        sum_cos = prefix_cos[right] - prefix_cos[left]
        sum_sin = prefix_sin[right] - prefix_sin[left]
        values = np.cos(c * x) * sum_cos + np.sin(c * x) * sum_sin
        return (np.pi / 4.0) * values / (n * bandwidth)

    raise ValueError(f"Kernel no soportado por el metodo rapido: {kernel}")


def evaluate_exponential_fast(data, x_grid, bandwidth):
    sorted_data = np.sort(np.asarray(data, dtype=float))
    n = float(sorted_data.size)
    x = np.asarray(x_grid, dtype=float)
    order = np.argsort(x)
    xs = x[order]

    left_sum = np.zeros_like(xs)
    acc = 0.0
    data_idx = 0
    previous_x = xs[0]
    for i, current_x in enumerate(xs):
        if i > 0:
            acc *= np.exp(-(current_x - previous_x) / bandwidth)
        while data_idx < sorted_data.size and sorted_data[data_idx] <= current_x:
            acc += np.exp((sorted_data[data_idx] - current_x) / bandwidth)
            data_idx += 1
        left_sum[i] = acc
        previous_x = current_x

    right_sum = np.zeros_like(xs)
    acc = 0.0
    data_idx = sorted_data.size - 1
    previous_x = xs[-1]
    for i in range(xs.size - 1, -1, -1):
        current_x = xs[i]
        if i < xs.size - 1:
            acc *= np.exp(-(previous_x - current_x) / bandwidth)
        while data_idx >= 0 and sorted_data[data_idx] > current_x:
            acc += np.exp((current_x - sorted_data[data_idx]) / bandwidth)
            data_idx -= 1
        right_sum[i] = acc
        previous_x = current_x

    out_sorted = 0.5 * (left_sum + right_sum) / (n * bandwidth)
    out = np.empty_like(out_sorted)
    out[order] = out_sorted
    return out


def evaluate_kde_fast(data, x_grid, bandwidth, kernel):
    data = np.asarray(data, dtype=float).ravel()
    data = data[np.isfinite(data)]
    if bandwidth <= 0:
        raise ValueError("El bandwidth debe ser positivo.")
    if kernel not in KERNELS:
        raise ValueError(f"Kernel desconocido: {kernel}")
    if kernel in FINITE_FAST_KERNELS:
        return evaluate_finite_support_fast(data, x_grid, bandwidth, kernel)
    if kernel == "exponential":
        return evaluate_exponential_fast(data, x_grid, bandwidth)
    return evaluate_kde_gaussian_fast(data, x_grid, bandwidth)


def compare_kernel_to_reference(data, x_grid, bandwidth, kernel, atol=1e-10, rtol_peak=1e-8):
    candidate = evaluate_kde_fast(data, x_grid, bandwidth, kernel)
    reference = evaluate_kde_reference(data, x_grid, bandwidth, kernel)
    max_abs = float(np.max(np.abs(candidate - reference)))
    peak = max(float(np.max(np.abs(reference))), np.finfo(float).tiny)
    rel_peak = float(max_abs / peak)
    candidate_norm, _ = normalize_conditional(candidate, x_grid)
    reference_norm, _ = normalize_conditional(reference, x_grid)
    norm_max_abs = float(np.max(np.abs(candidate_norm - reference_norm)))
    return {
        "kernel": kernel,
        "bandwidth": bandwidth,
        "error_abs_max": max_abs,
        "error_relativo_pico": rel_peak,
        "diferencia_curva": norm_max_abs,
        "pasa_validacion": bool(max_abs <= atol or rel_peak <= rtol_peak),
    }


In [ ]:
# Configuracion
DATA_PATH = "../Datos/otu_data_converted.csv"
CV_SUBSAMPLE = 1000
CV_FOLDS = 3
CV_BW_GRID = 8
GRID_SIZE = 1000
SELECTED_METHOD = "cv_por_kernel"

values, data_shape = load_otu_positives(DATA_PATH)

summary = pd.DataFrame([{
    "archivo": DATA_PATH,
    "filas": data_shape[0],
    "columnas": data_shape[1],
    "valores_positivos": len(values),
    "grid_usado": GRID_SIZE,
}])
display(summary)


In [ ]:
start = time.perf_counter()
h_scott = scott_h(values)
h_silverman = silverman_h(values)
cv_by_kernel_df, score_curves = estimate_cv_by_kernel(
    values,
    kernels=KERNELS,
    cv_folds=CV_FOLDS,
    n_subsample=CV_SUBSAMPLE,
    n_bw=CV_BW_GRID,
)
elapsed = time.perf_counter() - start

bandwidth_summary = cv_by_kernel_df.copy()
bandwidth_summary["bandwidth_scott"] = h_scott
bandwidth_summary["bandwidth_silverman"] = h_silverman
display(bandwidth_summary)
print(f"Tiempo de estimacion: {elapsed:.3f} s")


In [ ]:
kernel_bandwidths = dict(zip(cv_by_kernel_df["kernel"], cv_by_kernel_df["bandwidth_cv"]))

print("Valores para copiar en Calculo_Kernels.ipynb:")
print(f"selected_method = {SELECTED_METHOD!r}")
print("kernel_bandwidths = {")
for kernel, bandwidth in kernel_bandwidths.items():
    print(f"    {kernel!r}: {bandwidth:.12g},")
print("}")


In [ ]:
comparison_grid = make_log_grid(values, GRID_SIZE, max(kernel_bandwidths.values()))

fig, ax = plt.subplots(figsize=(10, 5))
for kernel in KERNELS:
    bandwidth = kernel_bandwidths[kernel]
    pdf = evaluate_kde_fast(values, comparison_grid, bandwidth, kernel)
    density, _ = normalize_conditional(pdf, comparison_grid)
    ax.plot(
        comparison_grid,
        density,
        linewidth=1.8,
        color=COLOR_MAP[kernel],
        label=f"{kernel} (h={bandwidth:.3g})",
    )

ax.set_xscale("log")
ax.set_xlabel("Valor OTU positivo")
ax.set_ylabel("Densidad condicional")
ax.set_title("KDE con bandwidth optimo por kernel")
ax.grid(True, alpha=0.25)
ax.legend(ncol=2, fontsize=9)
fig.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8), sharex=True)
for ax, kernel in zip(axes.ravel(), KERNELS):
    bw_grid, scores = score_curves[kernel]
    best_bw = kernel_bandwidths[kernel]
    ax.plot(bw_grid, scores, marker="o", linewidth=1.3, color=COLOR_MAP[kernel])
    ax.axvline(best_bw, color="black", linestyle="--", linewidth=1.0)
    ax.set_xscale("log")
    ax.set_title(f"{kernel} | h={best_bw:.3g}")
    ax.grid(True, alpha=0.25)

for ax in axes[-1, :]:
    ax.set_xlabel("Bandwidth")
for ax in axes[:, 0]:
    ax.set_ylabel("Score promedio")

fig.suptitle("Seleccion de bandwidth por kernel", fontsize=13)
fig.tight_layout()
plt.show()
